In [1]:
import torch

In [1]:
import gotrackit
import pandas as pd
import geopandas as gpd
import numpy as np
import networkx as nx 
import osmnx as ox
import matplotlib.pyplot as plt

In [2]:
# 将点边表转换为networkx图
def gdf2graph(nodes, links):
    links['key'] = 0
    nodes.set_index('id',inplace=True)
    links.set_index(['u','v','key'],inplace=True)
    return ox.graph_from_gdfs(nodes,links)

# 读取数据 点 边 区域边界 连接关系表
nodes = gpd.read_file('data/taicangNet/nodes.shp', encoding='utf-8')
links = gpd.read_file('data/taicangNet/links.shp', encoding='utf-8')
nodes.crs = 'epsg:3857'
links.crs = 'epsg:3857'

taicang = gpd.read_file('data/taicangNet/tcborder.shp', encoding='utf-8')
connections = pd.read_csv('data/rawdata/节点流向关系表20241228.csv', encoding='gbk')
connections.columns = ['from_link','id','to_link','direction','lane_cnt']
print(len(nodes),len(links),len(connections))
connections = connections[connections['from_link'].isin(links['id']) & connections['to_link'].isin(links['id'])]
print(len(nodes),len(links),len(connections))

# 生成路网图以检查连通性
G = gdf2graph(nodes.copy(), links.copy())
# 最大连通分量
largest_component = max(list(nx.weakly_connected_components(G)), key=len)
nodes = nodes[nodes['id'].isin(largest_component)]
links = links[(links['u'].isin(largest_component)) & (links['v'].isin(largest_component))]
connections = connections[connections['id'].isin(largest_component)]
print(len(nodes),len(links),len(connections))

4435 10378 28817
4435 10378 28552
4435 10378 28552


c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\osmnx\convert.py:306: UserWarning: Discarding the `gdf_nodes` 'geometry' column, though its values differ from the coordinates in the 'x' and 'y' columns.
  _validate_node_edge_gdfs(gdf_nodes, gdf_edges)


In [3]:
#节点
import xml.dom.minidom
from shapely.geometry import LineString
doc1 = xml.dom.minidom.Document() 
root = doc1.createElement('nodes')
doc1.appendChild(root)
for i in range(len(nodes)):
    node = doc1.createElement('node')
    node.setAttribute('id',str(nodes['id'].iloc[i]))
    node.setAttribute('x', str(nodes['x'].iloc[i]))
    node.setAttribute('y', str(nodes['y'].iloc[i]))
    node.setAttribute('type', str(nodes['type'].iloc[i]))
    root.appendChild(node)
fp1 = open('data/sumonet/taicang.nodes.xml', 'w', encoding='utf-8')  # 需要指定utf-8的文件编码格式，不然notepad中显示十六进制
doc1.writexml(fp1, indent='', addindent='\t', newl='\n', encoding='utf-8')
fp1.close()
#边
doc2 = xml.dom.minidom.Document()
root = doc2.createElement('links')
doc2.appendChild(root)
for i in range(len(links)):
    edge = doc2.createElement('edge')
    edge.setAttribute('id',str(links['id'].iloc[i]))
    edge.setAttribute('from',str(links['u'].iloc[i]))
    edge.setAttribute('to',str(links['v'].iloc[i]))
    edge.setAttribute('numLanes',str(int(links['numLanes'].iloc[i])))
    edge.setAttribute('length',str(links['length'].iloc[i]))
    # 提取几何信息
    geom = links['geometry'].iloc[i]
    if isinstance(geom, LineString):  # 确保几何是 LineString 类型
        # 将几何转换为 SUMO 的 shape 格式
        shape = " ".join([f"{coord[0]},{coord[1]}" for coord in geom.coords])
        edge.setAttribute('shape', shape)  # 添加 shape 属性
    root.appendChild(edge)
fp2 = open('data/sumonet/taicang.edges.xml', 'w', encoding='utf-8')  # 需要指定utf-8的文件编码格式，不然notepad中显示十六进制
doc2.writexml(fp2, indent='', addindent='\t', newl='\n', encoding='utf-8')
fp2.close()
# connections1
import xml.dom.minidom
doc3 = xml.dom.minidom.Document()
root = doc3.createElement('connections')
doc3.appendChild(root)
for i in range(len(connections)):
    fromedge = connections['from_link'].iloc[i]
    toedge = connections['to_link'].iloc[i]
    connection = doc3.createElement('connection')
    connection.setAttribute('from',str(fromedge))
    connection.setAttribute('to',str(toedge))
    root.appendChild(connection)
fp3 = open('data/sumonet/taicang.connections.xml', 'w', encoding='utf-8')  # 需要指定utf-8的文件编码格式，不然notepad中显示十六进制
doc3.writexml(fp3, indent='', addindent='\t', newl='\n', encoding='utf-8')
fp3.close()

In [4]:
!netconvert --node-files=data/sumonet/taicang.nodes.xml --edge-files=data/sumonet/taicang.edges.xml --connection-files=data/sumonet/taicang.connections.xml --output-file=data/sumonet/taicang1.net.xml --no-internal-links --log --error-log

Success.


In [5]:
# 转换格式
def nodeconvert(nodes):
    nodes1 = nodes.loc[:,['id','geometry']]
    nodes1.to_crs(epsg=4326, inplace=True)
    nodes1.columns = ['node_id','geometry']
    return nodes1
def linkconvert(links):
    links1 = links.loc[:,['id','u','v','length','geometry']]
    links1.to_crs(epsg=4326, inplace=True)
    links1.columns = ['link_id','from_node','to_node','length','geometry']
    links1['dir'] = 1
    return links1


In [6]:
import pickle # 读取轨迹数据
with open('data/traj/traj_list.pkl', 'rb') as file:
    traj_list = pickle.load(file)
move = pd.read_csv('data/traj/move.csv',index_col=0)

In [77]:
move['stime'] = pd.to_datetime(move['stime'])
move['etime'] = pd.to_datetime(move['etime'])
move['duration'] = (move['etime'] - move['stime']).dt.total_seconds() / 60  # 转换为分钟

In [7]:
# 构造轨迹数据表
for i in range(len(traj_list)):
    traj_list[i]['agent_id'] = i
gps_gdf = pd.concat(traj_list)
gps_gdf['lng'] = gps_gdf['geometry'].apply(lambda x: x.x)
gps_gdf['lat'] = gps_gdf['geometry'].apply(lambda x: x.y)
# 选取太仓范围内的轨迹数据
from shapely.ops import unary_union
taicang.crs = 'epsg:3857'
taicang = taicang.to_crs('epsg:4326')
taicang_boundary = unary_union(taicang.geometry)
gps_gdf = gps_gdf[gps_gdf.geometry.within(taicang_boundary)]

In [8]:
gps_gdf

,time,geometry,agent_id,lng,lat
47,2024-01-10 20:33:10,POINT (121.13 31.51306),2,121.129999,31.513060
49,2024-01-10 20:33:35,POINT (121.12817 31.51712),2,121.128171,31.517120
50,2024-01-10 20:34:25,POINT (121.12273 31.52664),2,121.122730,31.526639
51,2024-01-10 20:34:30,POINT (121.12228 31.52734),2,121.122277,31.527339
52,2024-01-10 20:35:20,POINT (121.1162 31.53508),2,121.116196,31.535081
...,...,...,...,...,...
20871168,2024-01-10 22:06:11,POINT (121.06709 31.58679),169610,121.067091,31.586791
20871169,2024-01-10 22:06:41,POINT (121.06116 31.58973),169610,121.061156,31.589727
20871170,2024-01-10 22:07:36,POINT (121.04887 31.59719),169610,121.048874,31.597190
20871171,2024-01-10 22:08:10,POINT (121.0433 31.60146),169610,121.043300,31.601457


In [278]:
# 轨迹OD点识别
gps_gdf['id_shift'] = gps_gdf['agent_id'].shift(1)
gps_gdf['origin'] = gps_gdf['agent_id'] - gps_gdf['id_shift']
gps_gdf['origin'] = gps_gdf['origin'].fillna(1)
gps_gdf['destination'] = gps_gdf['origin'].shift(-1)
gps_gdf['destination'] = gps_gdf['destination'].fillna(1)
gps_gdf = gps_gdf.drop(columns=['id_shift'])
Origin = gps_gdf[gps_gdf['origin'] == 1]
Destination = gps_gdf[gps_gdf['destination'] == 1]

In [279]:
Origin

,time,geometry,agent_id,lng,lat,origin,destination
47,2024-01-10 20:33:10,POINT (121.13 31.51306),2,121.129999,31.513060,1.0,0.0
1102,2024-01-10 09:55:13,POINT (121.02982 31.60972),8,121.029818,31.609725,1.0,0.0
1486,2024-01-10 19:33:00,POINT (121.13164 31.50904),14,121.131639,31.509040,1.0,0.0
1685,2024-01-10 22:15:56,POINT (121.14599 31.48661),17,121.145986,31.486608,1.0,0.0
2006,2024-01-10 12:01:14,POINT (121.13828 31.49737),22,121.138280,31.497367,1.0,0.0
...,...,...,...,...,...,...,...
20866871,2024-01-10 20:08:25,POINT (121.1319 31.50837),169562,121.131905,31.508366,1.0,0.0
20866913,2024-01-10 20:43:10,POINT (121.03404 31.60843),169563,121.034044,31.608425,1.0,0.0
20869966,2024-01-10 21:25:00,POINT (121.13483 31.40143),169598,121.134825,31.401426,1.0,0.0
20869991,2024-01-10 19:54:15,POINT (121.03806 31.60502),169599,121.038059,31.605023,1.0,0.0


In [280]:
Origin.to_file('data/Origin.shp',encoding='utf-8')
Destination.to_file('data/Destination.shp',encoding='utf-8')

C:\Users\dell\AppData\Local\Temp\ipykernel_10740\3308604319.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  Origin.to_file('data/Origin.shp',encoding='utf-8')
c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field time create as date field, though DateTime requested.
  ogr_write(
c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'destination' to 'destinatio'
  ogr_write(
C:\Users\dell\AppData\Local\Temp\ipykernel_10740\3308604319.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  Destination.to_file('data/Destination.shp',encoding='utf-8')
c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field time create as date field, though DateTime requested.
  ogr_write(
c:\Use

In [9]:
gps_df = gps_gdf[gps_gdf['agent_id']<20000]

In [18]:
from gotrackit.gps.GpsTrip import GpsPreProcess
grp = GpsPreProcess(gps_df=gps_gdf, use_multi_core=False)
# 调用trip_segmentations方法进行行程切分
# 切分后的数据会更新agent_id字段用以区分不同的出行旅程，原GPS表的agent_id会存储在origin_agent_id字段中
gps_trip = grp.trip_segmentations(group_gap_threshold=900, plain_crs='EPSG:32650', min_distance_threshold=10.0)
gps_trip

85410 vehicles, cutting group...


,time,agent_id,lng,lat,origin_agent_id
0,2024-01-10 20:33:10,41670,121.129999,31.513060,2
1,2024-01-10 20:33:35,41670,121.128171,31.517120,2
2,2024-01-10 20:34:25,41670,121.122730,31.526639,2
3,2024-01-10 20:34:30,41670,121.122277,31.527339,2
4,2024-01-10 20:35:20,41670,121.116196,31.535081,2
...,...,...,...,...,...
3638976,2024-01-10 22:06:11,41762,121.067091,31.586791,169610
3638977,2024-01-10 22:06:41,41762,121.061156,31.589727,169610
3638978,2024-01-10 22:07:36,41762,121.048874,31.597190,169610
3638979,2024-01-10 22:08:10,41762,121.043300,31.601457,169610


__init__ costs :0.07137298583984375 seconds!
AttributeError("'NoneType' object has no attribute 'coords'")


In [19]:
from gotrackit.map.Net import Net
from gotrackit.MapMatch import MapMatch
my_net = Net(link_gdf=linkconvert(links), node_gdf=nodeconvert(nodes), cut_off=2000.0, prj_cache=True) # 启用投影缓存
my_net.init_net()  # net初始化
# 实例化MapMatch类
mpm = MapMatch(net=my_net, flag_name='tc_sample', 
               gps_buffer=100, dense_gps=True, dense_interval=1000,
               use_sub_net=True, use_heading_inf=True, omitted_l=6.0,
               del_dwell=True, dwell_l_length=50.0, dwell_n=0,
               export_html=False, export_geo_res=False, use_gps_source=False,
               gps_radius=15.0, export_all_agents=False,
               out_fldr=r'data/output/match_visualization/tc_sample')

# execute函数返回三个结果:
# 第一个是匹配结果表、第二个是警告信息、第三个是错误信息
match_res, warn_info, error_info = mpm.multi_core_execute(gps_df=gps_gdf, core_num=4)
match_res.to_csv(r'data/output/match_res.csv', encoding='utf_8_sig', index=False)


using multiprocessing - 4 cores


In [20]:
match_res = pd.read_csv(r'data/output/match_res.csv')
match_res

,agent_id,seq,sub_seq,time,loc_type,link_id,from_node,to_node,lng,lat,prj_lng,prj_lat,dis_to_next,match_heading,route_dis
0,2,0,0,2024-01-10 20:33:10.000000000,s,32058505060,32058500001435,32058500001679,121.129999,31.513060,121.129994,31.513058,482.376646,338.95,30.983270
1,2,1,0,2024-01-10 20:33:35.000000000,s,32058505060,32058500001435,32058500001679,121.128171,31.517120,121.128089,31.517091,587.945641,338.95,513.359916
2,2,2,0,2024-01-10 20:34:00.000000000,d,32058501385,32058500001679,32058500001680,121.125451,31.521879,121.125578,31.521934,587.352531,334.43,249.205557
3,2,3,0,2024-01-10 20:34:25.000000000,s,32058504378,32058500001680,32058500001532,121.122730,31.526639,121.122604,31.526573,88.676008,329.60,270.858088
4,2,4,0,2024-01-10 20:34:30.000000000,s,32058504378,32058500001680,32058500001532,121.122277,31.527339,121.122118,31.527255,516.800398,329.60,359.534096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4049838,169610,23,0,2024-01-10 22:06:41.000000000,s,32058504158,32058500001515,32058500001322,121.061156,31.589727,121.061148,31.589714,715.337856,301.08,585.644962
4049839,169610,24,0,2024-01-10 22:07:08.500000000,d,32058501052,32058500001322,32058500000576,121.055015,31.593459,121.054820,31.593212,716.906563,305.16,487.882819
4049840,169610,25,0,2024-01-10 22:07:36.000000000,s,32058500450,32058500000576,32058500000575,121.048874,31.597190,121.048864,31.597181,709.664593,312.88,56.689382
4049841,169610,26,0,2024-01-10 22:08:10.000000000,s,32058500450,32058500000576,32058500000575,121.043300,31.601457,121.043295,31.601452,697.055035,312.88,766.353974
